<a href="https://colab.research.google.com/github/alexmando/NeuroSymbolic_CardGame/blob/mandolito/CardGame_SemanticLoss_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NEURO-SYMBOLIC AI — ASSIGNMENT 2026/2027**
## *Card Suit Game: Integrating Neural Learning and Semantic Loss*

---

### **1. PROJECT OBJECTIVE**

The objective of this project is to investigate how **symbolic knowledge can be integrated into neural network training through *Semantic Loss***, using the *Card Suit Game* as a controlled neuro-symbolic learning task.

Given a pair of playing card images ` (I_x, I_y) `, the model must simultaneously **recognize the suit of each card** and **predict the game outcome**. The game involves three suits — **Hearts, Clubs, and Spades** — governed by cyclic dominance rules:

- ♥ **Hearts** beats Clubs.
- ♣ **Clubs** beats Spades.
- ♠ **Spades** beats Hearts.
- Identical suits result in a **Draw**.

*Card ranks are completely disregarded.*

> **Research Question**
>
> *Can the integration of logical constraints into the neural learning objective improve prediction consistency while maintaining competitive classification accuracy?*

---

### **2. PROPOSED METHODOLOGY**

We adopt a **comparative experimental approach**, implementing two models with the *same neural architecture, training data, and experimental conditions*. Their only difference is the **optimization objective**.

| **NEURAL BASELINE** | **NEURO-SYMBOLIC MODEL** |
|:---|:---|
| Purely supervised learning | Supervised learning + Semantic Loss |
| Three Cross Entropy terms | Three Cross Entropy + four Semantic Loss terms |
| No explicit logical constraints | Explicit logical regularization |

#### **A. Neural Baseline — Purely Supervised Learning**

The baseline learns exclusively from labeled examples, minimizing the sum of three **Cross Entropy losses**:

`L_neural = L_x + L_y + L_o`

#### **B. Neuro-Symbolic Model — Semantic Loss**

The neuro-symbolic model incorporates two categories of logical constraints:

- **Exactly-One Constraints (αEO):** exactly one suit must be assigned to each card, and exactly one game outcome must be predicted.
- **Game Rules (αRules):** the predicted outcome must be consistent with the suits of both cards and their dominance relation.

The network produces **nine output logits**, organized into three groups of three: *first card, second card, and game outcome*. Their sigmoid transformations provide the probabilities associated with the nine Boolean propositions used by Semantic Loss.

**Complete Optimization Objective**

```text
L_total = omega_1 * (L_x + L_y + L_o)
        + omega_2 * (EO_x + EO_y + EO_o)
        + omega_3 * SL_xy
```

The hyperparameters **`omega_1`, `omega_2`, and `omega_3`** control the contribution of:

- *Supervised learning* — classification performance.
- *Exactly-One constraints* — logical exclusivity of the predictions.
- *Game-rule consistency* — compatibility between card suits and game outcome.

---

### **3. EXPERIMENTAL PROTOCOL**

The experimental investigation is structured around **five main steps**:

1. **Baseline Training:** establish the performance of the purely neural model.
2. **Neuro-Symbolic Training:** optimize the model using the complete objective function.
3. **Ablation Study:** evaluate the contribution of logical regularization by comparing the two approaches.
4. **Hyperparameter Tuning:** investigate different combinations of loss weights using the *validation set*.
5. **Final Evaluation:** compare predictive accuracy, logical constraint satisfaction, game-rule consistency, and training time on the *test set*.

Both models use **identical data partitions and card pairs** to ensure a fair experimental comparison.

The dataset is obtained from the **Cards Image Dataset – Classification (Kaggle)**, retaining only *Hearts, Clubs, and Spades*, while excluding *Diamonds and Jokers*.

Additional experiments with **20% and 50% of the training data** are retained to investigate the effect of data availability on model performance.

---

### **4. EXPECTED CONTRIBUTION**

The project aims to assess whether **Semantic Loss provides an effective mechanism for integrating symbolic knowledge into data-driven learning**.

Rather than relying exclusively on statistical patterns, the neuro-symbolic model explicitly penalizes predictions that violate the *logical structure of the game*.

The final comparison will highlight the impact of logical regularization on:

- **Classification Performance**
- **Logical Consistency**
- **Computational Efficiency**

The theoretical component will also examine the foundations of *Semantic Loss* and the implementation of logical constraints through **brute-force enumeration of Boolean assignments**.


In [ ]:
%pip install pysdd

!git clone https://github.com/lucadiliello/semantic-loss-pytorch
!sed -i 's/batch_size = probabilities.size(-1)/batch_size = probabilities.size(0)/g' /content/semantic-loss-pytorch/semantic_loss_pytorch/semantic_loss.py

%pip install ./semantic-loss-pytorch

In [ ]:
!pip install -q kagglehub pandas scikit-learn matplotlib pillow

import random
import shutil
from pathlib import Path

import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split

# Reproducibility and label conventions
SEED = 42
ALLOWED_SUITS = ["hearts", "clubs", "spades"]
SUIT_TO_ID = {"hearts": 0, "clubs": 1, "spades": 2}

# Outcome order follows the assignment: {1, x, 2}
# 1 = Player 1 wins, x = Draw, 2 = Player 2 wins
OUTCOME_TO_ID = {"player1": 0, "draw": 1, "player2": 2}

print("Downloading the Kaggle card dataset...")
dataset_path = Path(kagglehub.dataset_download(
    "gpiosenka/cards-image-datasetclassification"
))
output_dir = Path("/content/cards_filtered")
print(f"Original dataset: {dataset_path}")

# Remove a previous filtered copy to avoid keeping stale or duplicate images.
if output_dir.exists():
    shutil.rmtree(output_dir)

# Keep only the three allowed suits in the original train and test folders.
image_extensions = {".jpg", ".jpeg", ".png"}
for split in ["train", "test"]:
    split_src = dataset_path / split
    split_dst = output_dir / split

    if not split_src.exists():
        raise FileNotFoundError(f"Expected dataset split not found: {split_src}")

    for suit in ALLOWED_SUITS:
        (split_dst / suit).mkdir(parents=True, exist_ok=True)

    for card_folder in split_src.iterdir():
        if not card_folder.is_dir():
            continue

        folder_name = card_folder.name.lower()

        # Assignment requirement: exclude Diamonds and Jokers.
        if "joker" in folder_name or "diamonds" in folder_name:
            continue

        target_suit = next(
            (suit for suit in ALLOWED_SUITS if suit in folder_name),
            None
        )
        if target_suit is None:
            continue

        for image_file in card_folder.iterdir():
            if image_file.is_file() and image_file.suffix.lower() in image_extensions:
                destination = split_dst / target_suit / f"{card_folder.name}_{image_file.name}"
                shutil.copy2(image_file, destination)

print(f"Filtered images saved in: {output_dir}")

# Map every pair of suits to the correct game outcome.
def determine_outcome(suit1, suit2):
    if suit1 == suit2:
        outcome = "draw"
    elif (suit1 == "hearts" and suit2 == "clubs") or \
         (suit1 == "clubs" and suit2 == "spades") or \
         (suit1 == "spades" and suit2 == "hearts"):
        outcome = "player1"
    else:
        outcome = "player2"

    return outcome, OUTCOME_TO_ID[outcome]


# Collect original images by suit
def collect_images(split_dir):
    images_by_suit = {}

    for suit in ALLOWED_SUITS:
        images_by_suit[suit] = sorted([
            path for path in (split_dir / suit).glob("*")
            if path.is_file()
            and path.suffix.lower() in image_extensions
        ])

        if not images_by_suit[suit]:
            raise RuntimeError(f"No images found for {suit}")

    return images_by_suit


# Collect original training and test images
original_train_images = collect_images(output_dir / "train")
test_image_pool = collect_images(output_dir / "test")

train_image_pool = {}
val_image_pool = {}

# Split original images BEFORE generating pairs
for suit in ALLOWED_SUITS:

    train_images, val_images = train_test_split(
        original_train_images[suit],
        test_size=0.15,
        random_state=SEED
    )

    train_image_pool[suit] = train_images
    val_image_pool[suit] = val_images


# Generate pairs from a specific image pool
def create_pairs(images_by_suit, num_pairs, seed=SEED):

    rng = random.Random(seed)
    pairs = []

    for _ in range(num_pairs):

        suit1 = rng.choice(ALLOWED_SUITS)
        suit2 = rng.choice(ALLOWED_SUITS)

        img1 = rng.choice(images_by_suit[suit1])
        img2 = rng.choice(images_by_suit[suit2])

        outcome_str, outcome_id = determine_outcome(
            suit1, suit2
        )

        pairs.append({
            "img1_path": str(img1),
            "img2_path": str(img2),
            "suit1": suit1,
            "suit2": suit2,
            "outcome_str": outcome_str,
            "label": outcome_id
        })

    return pd.DataFrame(pairs)


# Generate independent datasets
df_train_full = create_pairs(
    train_image_pool, 2550, SEED
)

df_val = create_pairs(
    val_image_pool, 450, SEED + 1
)

df_test = create_pairs(
    test_image_pool, 1000, SEED + 2
)


# Generate smaller training subsets
df_train_50, _ = train_test_split(
    df_train_full,
    train_size=0.50,
    random_state=SEED,
    stratify=df_train_full["label"]
)

df_train_20, _ = train_test_split(
    df_train_full,
    train_size=0.20,
    random_state=SEED,
    stratify=df_train_full["label"]
)


# Save manifests
csv_dir = Path("/content/manifests")
csv_dir.mkdir(parents=True, exist_ok=True)

manifests = {
    "train_100.csv": df_train_full,
    "train_50.csv": df_train_50,
    "train_20.csv": df_train_20,
    "val.csv": df_val,
    "test.csv": df_test,
}

for filename, dataframe in manifests.items():
    dataframe.to_csv(csv_dir / filename, index=False)

print("--- MANIFEST GENERATION COMPLETED ---")

for filename, dataframe in manifests.items():
    print(f"{filename}: {len(dataframe)} pairs")

print("Outcome label mapping:", OUTCOME_TO_ID)
print("Suit label mapping:", SUIT_TO_ID)
print("Manifest folder:", csv_dir)

**Data Exploration and Class Balance Verification**


The next cell loads the generated training manifest and checks the distribution of the three outcomes. It also displays a few image pairs together with their suit labels and the expected game result. This is a data-integrity check: before training, we verify that the filtering, pair generation, and game rules have been applied correctly.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Load the full training manifest.
csv_path = "/content/manifests/train_100.csv"
df = pd.read_csv(csv_path)

OUTCOME_ORDER = ["player1", "draw", "player2"]

print("DATA EXPLORATION: CARD DUEL DATASET")
print(f"Total generated games: {len(df)}")

# Show all three classes in the same order as the output labels: {1, x, 2}.
class_counts = df["outcome_str"].value_counts().reindex(OUTCOME_ORDER, fill_value=0)
print("\nOutcome distribution:")
print(class_counts)

fig, ax = plt.subplots(figsize=(6, 4))
class_counts.plot(kind="bar", ax=ax)
ax.set_title("Distribution of Game Outcomes")
ax.set_ylabel("Number of Pairs")
ax.set_xlabel("Outcome")
ax.set_xticklabels(class_counts.index, rotation=0)
plt.tight_layout()
plt.show()


def show_sample_matches(dataframe, num_samples=4):
    if dataframe.empty:
        print("The manifest is empty; there are no examples to display.")
        return

    num_samples = min(num_samples, len(dataframe))
    sample_df = dataframe.sample(n=num_samples, random_state=SEED).reset_index(drop=True)

    fig, axes = plt.subplots(num_samples, 2, figsize=(8, 3 * num_samples), squeeze=False)
    fig.suptitle("Examples of Generated Card Games", fontsize=15)

    for i, row in sample_df.iterrows():
        with Image.open(row["img1_path"]) as image1:
            axes[i, 0].imshow(image1.convert("RGB"))
        axes[i, 0].set_title(f"Player 1: {row['suit1'].capitalize()}")
        axes[i, 0].axis("off")

        with Image.open(row["img2_path"]) as image2:
            axes[i, 1].imshow(image2.convert("RGB"))
        axes[i, 1].set_title(f"Player 2: {row['suit2'].capitalize()}")
        axes[i, 1].axis("off")

        print(
            f"Game {i + 1}: {row['suit1']} vs {row['suit2']} "
            f"-> {row['outcome_str']} (label={row['label']})"
        )

    plt.tight_layout()
    plt.show()


show_sample_matches(df, num_samples=4)


# **EVALUATION STRATEGY — NEURAL BASELINE VS. SEMANTIC LOSS**

---

### **1. COMPARATIVE EXPERIMENTAL APPROACH**

The experimental objective is to evaluate the contribution of **logical regularization** by comparing two versions of the same multi-output neural network.

Both models share the **same architecture, training data, labels, and data partitions**. The only difference is the loss function used during optimization.

| **NEURAL BASELINE** | **SEMANTIC LOSS MODEL** |
|:---|:---|
| Purely supervised learning | Neuro-symbolic learning |
| Three Cross Entropy terms | Cross Entropy + Semantic Loss |
| No explicit logical constraints | Exactly-One and game-rule constraints |

### **2. NEURAL BASELINE — SUPERVISED LEARNING**

The baseline learns the three classification tasks exclusively from labeled examples, minimizing the sum of three standard Cross Entropy losses:

```text
L_neural = L_x + L_y + L_o
```

These terms correspond to the classification of the **first card, second card, and game outcome**, respectively.

### **3. NEURO-SYMBOLIC MODEL — SEMANTIC REGULARIZATION**

The second model extends the supervised objective by introducing **four Semantic Loss terms** that encode the logical knowledge of the game:

- **EO_x, EO_y, EO_o:** enforce the *Exactly-One* constraint, requiring exactly one active class for each card and the game outcome.
- **SL_xy:** enforces consistency between the two card suits and the predicted game outcome according to the circular dominance rules.

The complete optimization objective is:

```text
L_total = omega_1 * (L_x + L_y + L_o)
        + omega_2 * (EO_x + EO_y + EO_o)
        + omega_3 * SL_xy
```

The hyperparameters **`omega_1`, `omega_2`, and `omega_3`** control the relative contribution of supervised learning and logical regularization.

### **4. SEMANTIC LOSS COMPUTATION**

The network produces nine output logits, which are processed differently according to their purpose:

- **Supervised learning:** raw logits are used with standard multi-class Cross Entropy.
- **Logical regularization:** sigmoid transformations provide nine independent Bernoulli probabilities, corresponding to the Boolean propositions used by Semantic Loss.

Using independent sigmoid probabilities is essential because the Exactly-One constraints must be learned through logical regularization rather than being automatically imposed by a three-class softmax.

To calculate the probability of satisfying each logical formula, we employ **brute-force enumeration** of all possible Boolean assignments:

`2^9 = 512 possible assignments`

The valid assignments are identified according to the formalized logical constraints, and their probabilities are combined to compute the corresponding Semantic Loss.

This approach is computationally feasible for the present problem and is explicitly permitted by the assignment. **No ASP or DeepProbLog inference engine is used.**

### **5. EVALUATION & MODEL SELECTION**

The two approaches are evaluated using the following metrics:

- **Suit Classification Accuracy:** accuracy for both card images.
- **Outcome Accuracy:** accuracy in predicting the game result.
- **Logical Constraint Satisfaction:** percentage of predictions satisfying the Exactly-One and game-rule constraints.
- **Training Time:** computational cost of model optimization.

The **validation set** is used for hyperparameter selection and model configuration, while the **test set** is reserved exclusively for the final performance comparison.

In [4]:
from pysdd.sdd import Vtree, SddManager
from graphviz import Source
from semantic_loss_pytorch import SemanticLoss
from functools import reduce
import operator
import os

In [5]:
# vtree initialization
vtree = Vtree(9)
manager = SddManager.from_vtree(vtree)

In [6]:
# SDD for the exactly-one constraint

literals = [manager.literal(i) for i in range(1, 10)]

sdd_exactly_one_list = []

x_card_exactly_one = []
for n in range(3):
    clause_variables = [literals[i] if i==n else ~literals[i] for i in range(3)]
    x_card_exactly_one.append(reduce(operator.and_, clause_variables))

y_card_exactly_one = []
for n in range(3, 6):
    clause_variables = [literals[i] if i==n else ~literals[i] for i in range(3, 6)]
    y_card_exactly_one.append(reduce(operator.and_, clause_variables))

output_exactly_one = []
for n in range(6, 9):
    clause_variables = [literals[i] if i==n else ~literals[i] for i in range(6, 9)]
    output_exactly_one.append(reduce(operator.and_, clause_variables))

for x_model in x_card_exactly_one:
    for y_model in y_card_exactly_one:
        for z_model in output_exactly_one:
            sdd_exactly_one_list.append(x_model & y_model & z_model)

sdd_exactly_one = reduce(operator.or_, sdd_exactly_one_list)

for modello in sdd_exactly_one.models():
    print(modello)

# chart = Source(sdd_exactly_one.dot())
# chart

print(sdd_exactly_one.wmc(log_mode=False).propagate())

os.makedirs('loss', exist_ok=True)
manager.save(b'loss/sdd_exactly_one.sdd', sdd_exactly_one)
vtree.save(b'loss/vtree_exactly_one.vtree')

{1: 0, 2: 0, 3: 1, 4: 0, 5: 1, 6: 0, 7: 0, 8: 1, 9: 0}
{1: 0, 2: 0, 3: 1, 4: 0, 5: 1, 6: 0, 7: 0, 8: 0, 9: 1}
{1: 0, 2: 0, 3: 1, 4: 0, 5: 1, 6: 0, 7: 1, 8: 0, 9: 0}
{1: 0, 2: 0, 3: 1, 4: 0, 5: 0, 6: 1, 7: 0, 8: 1, 9: 0}
{1: 0, 2: 0, 3: 1, 4: 0, 5: 0, 6: 1, 7: 0, 8: 0, 9: 1}
{1: 0, 2: 0, 3: 1, 4: 0, 5: 0, 6: 1, 7: 1, 8: 0, 9: 0}
{1: 1, 2: 0, 3: 0, 4: 0, 5: 1, 6: 0, 7: 0, 8: 1, 9: 0}
{1: 1, 2: 0, 3: 0, 4: 0, 5: 1, 6: 0, 7: 0, 8: 0, 9: 1}
{1: 1, 2: 0, 3: 0, 4: 0, 5: 1, 6: 0, 7: 1, 8: 0, 9: 0}
{1: 1, 2: 0, 3: 0, 4: 0, 5: 0, 6: 1, 7: 0, 8: 1, 9: 0}
{1: 1, 2: 0, 3: 0, 4: 0, 5: 0, 6: 1, 7: 0, 8: 0, 9: 1}
{1: 1, 2: 0, 3: 0, 4: 0, 5: 0, 6: 1, 7: 1, 8: 0, 9: 0}
{1: 0, 2: 1, 3: 0, 4: 0, 5: 1, 6: 0, 7: 0, 8: 1, 9: 0}
{1: 0, 2: 1, 3: 0, 4: 0, 5: 1, 6: 0, 7: 0, 8: 0, 9: 1}
{1: 0, 2: 1, 3: 0, 4: 0, 5: 1, 6: 0, 7: 1, 8: 0, 9: 0}
{1: 0, 2: 1, 3: 0, 4: 0, 5: 0, 6: 1, 7: 0, 8: 1, 9: 0}
{1: 0, 2: 1, 3: 0, 4: 0, 5: 0, 6: 1, 7: 0, 8: 0, 9: 1}
{1: 0, 2: 1, 3: 0, 4: 0, 5: 0, 6: 1, 7: 1, 8: 0, 9: 0}
{1: 0, 2: 

In [7]:
# SDD for the rules constraint

p1_hearts, p1_clubs, p1_spades = literals[0], literals[1], literals[2]
p2_hearts, p2_clubs, p2_spades = literals[3], literals[4], literals[5]
out_p1, out_draw, out_p2 = literals[6], literals[7], literals[8]

p1_wins = (p1_hearts & p2_clubs) | (p1_clubs & p2_spades) | (p1_spades & p2_hearts)
draw = (p1_hearts & p2_hearts) | (p1_clubs & p2_clubs) | (p1_spades & p2_spades)
p2_wins = (p1_hearts & p2_spades) | (p1_clubs & p2_hearts) | (p1_spades & p2_clubs)

game_rules_list = [
    (~p1_wins | out_p1)   & (~out_p1   | p1_wins),
    (~draw    | out_draw) & (~out_draw | draw),
    (~p2_wins | out_p2)   & (~out_p2   | p2_wins),
]

sdd_game_rules = reduce(operator.and_, game_rules_list)

for modello in sdd_game_rules.models():
    print(modello)

# chart = Source(sdd_game_rules.dot())
# chart

print(sdd_game_rules.wmc(log_mode=False).propagate())

os.makedirs('loss', exist_ok=True)
manager.save(b'loss/sdd_game_rules.sdd', sdd_game_rules)
vtree.save(b'loss/vtree_game_rules.vtree')

{1: 1, 2: 1, 3: 1, 4: 0, 5: 0, 6: 1, 7: 1, 8: 1, 9: 1}
{1: 1, 2: 1, 3: 1, 4: 0, 5: 1, 6: 0, 7: 1, 8: 1, 9: 1}
{1: 1, 2: 1, 3: 1, 4: 0, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1}
{1: 1, 2: 1, 3: 1, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0}
{1: 1, 2: 1, 3: 0, 4: 0, 5: 0, 6: 1, 7: 1, 8: 0, 9: 1}
{1: 1, 2: 1, 3: 0, 4: 0, 5: 1, 6: 0, 7: 1, 8: 1, 9: 0}
{1: 1, 2: 1, 3: 0, 4: 0, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1}
{1: 1, 2: 1, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0}
{1: 1, 2: 1, 3: 0, 4: 1, 5: 0, 6: 0, 7: 0, 8: 1, 9: 1}
{1: 1, 2: 1, 3: 0, 4: 1, 5: 0, 6: 1, 7: 1, 8: 1, 9: 1}
{1: 1, 2: 1, 3: 0, 4: 1, 5: 1, 6: 0, 7: 1, 8: 1, 9: 1}
{1: 1, 2: 1, 3: 0, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1}
{1: 0, 2: 1, 3: 1, 4: 0, 5: 1, 6: 0, 7: 0, 8: 1, 9: 1}
{1: 0, 2: 1, 3: 1, 4: 0, 5: 0, 6: 1, 7: 1, 8: 1, 9: 0}
{1: 0, 2: 1, 3: 1, 4: 0, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1}
{1: 0, 2: 1, 3: 1, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0}
{1: 0, 2: 1, 3: 0, 4: 0, 5: 1, 6: 1, 7: 1, 8: 1, 9: 0}
{1: 0, 2: 1, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0}
{1: 0, 2: 

In [8]:
sl_exactly_one = SemanticLoss('loss/sdd_exactly_one.sdd', 'loss/vtree_exactly_one.vtree')
sl_game_rules = SemanticLoss('loss/sdd_game_rules.sdd', 'loss/vtree_game_rules.vtree')

### **5. Data Preparation and PyTorch DataLoaders**
This section implements the custom `CardDuelDataset` to load image pairs and their labels from the CSV manifests, applying necessary transformations (resize to 128x128 and ImageNet normalization).

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
from pathlib import Path


tensor_cache = {}

def get_image_tensor(path, transform):
    if path not in tensor_cache:
        with Image.open(path) as img:
            tensor_cache[path] = transform(img.convert("RGB"))
    return tensor_cache[path]

# 1. Define Transforms
# Resizing to 128x128 and normalizing with ImageNet stats as required
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Define Dataset Class
class CardDuelDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.transform = transform
        # Suit to integer mapping
        self.suit_to_id = {"hearts": 0, "clubs": 1, "spades": 2}

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        row = self.data_frame.iloc[idx]

        # # Load images and convert to RGB
        # image1 = Image.open(row['img1_path']).convert('RGB')
        # image2 = Image.open(row['img2_path']).convert('RGB')

        # # Apply transformations
        # if self.transform:
        #     image1 = self.transform(image1)
        #     image2 = self.transform(image2)

        image1 = get_image_tensor(row['img1_path'], self.transform)
        image2 = get_image_tensor(row['img2_path'], self.transform)

        # Extract and encode labels
        suit1_label = self.suit_to_id[row['suit1']]
        suit2_label = self.suit_to_id[row['suit2']]
        outcome_label = row['label'] # Outcome is already an integer in the manifest

        # Return image tensors and the three labels
        return image1, image2, suit1_label, suit2_label, outcome_label

# 3. Initialize Datasets and DataLoaders
manifest_dir = Path("/content/manifests")
batch_size = 32

datasets = {
    "train_100": CardDuelDataset(manifest_dir / "train_100.csv", transform=transform),
    "train_50": CardDuelDataset(manifest_dir / "train_50.csv", transform=transform),
    "train_20": CardDuelDataset(manifest_dir / "train_20.csv", transform=transform),
    "val": CardDuelDataset(manifest_dir / "val.csv", transform=transform),
    "test": CardDuelDataset(manifest_dir / "test.csv", transform=transform)
}

# Create DataLoaders with shuffle=True only for training sets
dataloaders = {
    "train_100": DataLoader(datasets["train_100"], batch_size=batch_size, shuffle=True),
    "train_50": DataLoader(datasets["train_50"], batch_size=batch_size, shuffle=True),
    "train_20": DataLoader(datasets["train_20"], batch_size=batch_size, shuffle=True),
    "val": DataLoader(datasets["val"], batch_size=batch_size, shuffle=False),
    "test": DataLoader(datasets["test"], batch_size=batch_size, shuffle=False)
}

all_paths = set()
for ds in datasets.values():
    all_paths.update(ds.data_frame["img1_path"])
    all_paths.update(ds.data_frame["img2_path"])

for p in all_paths:
    get_image_tensor(p, transform)

print("Data pipeline ready!")
print(f"Batches in train_100: {len(dataloaders['train_100'])}")
print(f"Batches in val: {len(dataloaders['val'])}")
print(f"Batches in test: {len(dataloaders['test'])}")

Data pipeline ready!
Batches in train_100: 80
Batches in val: 15
Batches in test: 32


### **6. Neural Architecture (Siamese Multi-Head CNN)**
To prevent overfitting on this controlled classification task and reduce computational overhead, instead of using heavy pre-trained backbones, we use a lightweight, custom **Siamese Convolutional Neural Network** (~130k parameters).

Both card images (`128x128` RGB) are processed through the same shared 3-block convolutional feature extractor with Batch Normalization, Max Pooling, and Dropout regularization. The resulting 128-dimensional embeddings are then routed to three classification heads to produce **9 output logits** ordered as required:
* `[0:3]`: First card suit (Hearts, Clubs, Spades)
* `[3:6]`: Second card suit (Hearts, Clubs, Spades)
* `[6:9]`: Game outcome (Player 1, Draw, Player 2)

In [10]:
import torch
import torch.nn as nn

class CardNeuroSymbolicModel(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super(CardNeuroSymbolicModel, self).__init__()

        # 1. Lightweight Shared Feature Extractor (Siamese Backbone)
        # Input shape: [batch_size, 3, 128, 128]
        self.feature_extractor = nn.Sequential(
            # Block 1: 128x128 -> 64x64
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2: 64x64 -> 32x32
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3: 32x32 -> 16x16
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Global Average Pooling reduces [batch_size, 64, 16, 16] -> [batch_size, 64, 1, 1]
            # nn.AdaptiveAvgPool2d((1, 1)),
            nn.AdaptiveMaxPool2d((1, 1)),
            nn.Flatten()
        )

        # Compact projection layer per image (64 -> 64) with Dropout
        self.img_fc = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate)
        )

        # # 2. Multi-Head Classifier
        # # Head A: Suit of Card 1 (Hearts, Clubs, Spades)
        # self.fc_suit1 = nn.Linear(64, 3)

        # # Head B: Suit of Card 2 (Hearts, Clubs, Spades)
        # self.fc_suit2 = nn.Linear(64, 3)

        self.fc_suit = nn.Linear(64, 3)

        # Head C: Game Outcome (Player 1, Draw, Player 2)
        # Combines embeddings from both cards (64 + 64 = 128)
        self.fc_outcome = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Linear(64, 3)
        )

    def forward(self, img1, img2):
        # Extract compact embeddings for both cards using shared weights
        feat1 = self.img_fc(self.feature_extractor(img1))
        feat2 = self.img_fc(self.feature_extractor(img2))

        # # Predict individual card suits
        # logits_suit1 = self.fc_suit1(feat1)
        # logits_suit2 = self.fc_suit2(feat2)

        logits_suit1 = self.fc_suit(feat1)
        logits_suit2 = self.fc_suit(feat2)

        # Concatenate embeddings and predict game outcome
        combined_feats = torch.cat((feat1, feat2), dim=1)
        logits_outcome = self.fc_outcome(combined_feats)

        # Output shape: [batch_size, 9] -> [Suit1 (3), Suit2 (3), Outcome (3)]
        return torch.cat((logits_suit1, logits_suit2, logits_outcome), dim=1)

# Quick verification of parameter count and output shape
if __name__ == "__main__":
    model = CardNeuroSymbolicModel()
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    dummy_img1 = torch.randn(32, 3, 128, 128)
    dummy_img2 = torch.randn(32, 3, 128, 128)
    out = model(dummy_img1, dummy_img2)

    print(f"Lightweight CNN initialized! Trainable parameters: {total_params:,}")
    print(f"Output shape: {out.shape} (Expected: [32, 9])")

Lightweight CNN initialized! Trainable parameters: 36,614
Output shape: torch.Size([32, 9]) (Expected: [32, 9])


### **7. Training & Evaluation Pipeline (Neural Baseline)**
This section implements the training and validation loop along with the full suite of evaluation metrics required by the experimental protocol:
* **Classification Accuracy:** Top-1 accuracy for the first card suit, second card suit, and game outcome.
* **Exactly-One Satisfaction (`EO_sat`):** Percentage of samples where thresholded Bernoulli probabilities (`sigmoid(logits) > 0.5`) activate strictly one class per group.
* **Rule Consistency (`Rule_cons`):** Percentage of samples where the predicted suits and predicted outcome are logically consistent with the cyclic dominance rules.
* **Training Time:** Total wall-clock time required for optimization.

The training function includes a modular hook (`semantic_loss_fn`) so that the brute-force Semantic Loss terms (`EO` and `SL_xy`) can be seamlessly integrated for the Neuro-Symbolic experiments without altering the training pipeline.

In [11]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set device (GPU if available on Colab, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def compute_batch_metrics(logits, target_x, target_y, target_o):
    """
    Computes classification accuracies and logical constraint satisfaction
    for a batch of 9-logit predictions.
    """
    batch_size = logits.size(0)

    # 1. Top-1 Predictions (argmax) for Classification Accuracy & Rule Consistency
    pred_x = logits[:, 0:3].argmax(dim=1)
    pred_y = logits[:, 3:6].argmax(dim=1)
    pred_o = logits[:, 6:9].argmax(dim=1)

    correct_x = (pred_x == target_x).sum().item()
    correct_y = (pred_y == target_y).sum().item()
    correct_o = (pred_o == target_o).sum().item()

    # 2. Rule Consistency on Top-1 Predictions
    # Check if (pred_x, pred_y) logically entails pred_o according to game rules:
    # 0=Hearts, 1=Clubs, 2=Spades | Outcome: 0=Player1, 1=Draw, 2=Player2
    expected_o = torch.full_like(pred_o, 2) # Default to Player 2 wins
    expected_o[pred_x == pred_y] = 1        # Draw
    p1_wins_mask = ((pred_x == 0) & (pred_y == 1)) | \
                   ((pred_x == 1) & (pred_y == 2)) | \
                   ((pred_x == 2) & (pred_y == 0))
    expected_o[p1_wins_mask] = 0            # Player 1 wins

    rule_consistent = (pred_o == expected_o).sum().item()

    # 3. Exactly-One Satisfaction on Sigmoid Probabilities (threshold = 0.5)
    # probs = torch.sigmoid(logits)
    probs = torch.softmax(logits.reshape(-1, 3, 3), dim=-1).reshape(-1, 9)
    bin_preds = (probs > 0.5).int()

    eo_x = (bin_preds[:, 0:3].sum(dim=1) == 1)
    eo_y = (bin_preds[:, 3:6].sum(dim=1) == 1)
    eo_o = (bin_preds[:, 6:9].sum(dim=1) == 1)
    eo_all = (eo_x & eo_y & eo_o).sum().item()

    return {
        "correct_x": correct_x,
        "correct_y": correct_y,
        "correct_o": correct_o,
        "rule_consistent": rule_consistent,
        "eo_all": eo_all,
        "total": batch_size
    }

def evaluate_model(model, dataloader, semantic_loss_fn=None, w1=1.0, w2=0.0, w3=0.0):
    """Evaluates the model on a validation or test DataLoader."""
    model.eval()
    total_loss = 0.0
    agg = {"correct_x": 0, "correct_y": 0, "correct_o": 0, "rule_consistent": 0, "eo_all": 0, "total": 0}

    with torch.no_grad():
        for img1, img2, t_x, t_y, t_o in dataloader:
            img1, img2 = img1.to(device), img2.to(device)
            t_x, t_y, t_o = t_x.to(device), t_y.to(device), t_o.to(device)

            logits = model(img1, img2)

            # Supervised Cross Entropy Loss: L_neural = L_x + L_y + L_o
            l_neural = F.cross_entropy(logits[:, 0:3], t_x) + \
                       F.cross_entropy(logits[:, 3:6], t_y) + \
                       F.cross_entropy(logits[:, 6:9], t_o)

            loss = w1 * l_neural

            # Hook for Alessandro's Semantic Loss function
            if semantic_loss_fn is not None:
                eo_loss, sl_xy = semantic_loss_fn(logits)
                loss = loss + w2 * eo_loss + w3 * sl_xy

            total_loss += loss.item() * img1.size(0)

            batch_m = compute_batch_metrics(logits, t_x, t_y, t_o)
            for k in agg:
                agg[k] += batch_m[k]

    n = agg["total"]
    return {
        "loss": total_loss / n,
        "acc_x": agg["correct_x"] / n * 100.0,
        "acc_y": agg["correct_y"] / n * 100.0,
        "acc_o": agg["correct_o"] / n * 100.0,
        "eo_sat": agg["eo_all"] / n * 100.0,
        "rule_cons": agg["rule_consistent"] / n * 100.0
    }

def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3,
                semantic_loss_fn=None, w1=1.0, w2=0.0, w3=0.0, save_path="best_model.pth"):
    """Trains either the Neural Baseline or the Neuro-Symbolic Model."""
    model = model.to(device)
    # optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-5
    )

    history = {
        "train_loss": [], "val_loss": [],
        "val_acc_x": [], "val_acc_y": [], "val_acc_o": [],
        "val_eo_sat": [], "val_rule_cons": [], "lr": []
    }
    best_val_loss = float("inf")

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        total_samples = 0

        for img1, img2, t_x, t_y, t_o in train_loader:
            img1, img2 = img1.to(device), img2.to(device)
            t_x, t_y, t_o = t_x.to(device), t_y.to(device), t_o.to(device)

            optimizer.zero_grad()
            logits = model(img1, img2)

            # Supervised Cross Entropy Loss
            l_neural = F.cross_entropy(logits[:, 0:3], t_x) + \
                       F.cross_entropy(logits[:, 3:6], t_y) + \
                       F.cross_entropy(logits[:, 6:9], t_o)

            loss = w1 * l_neural

            # Hook for Semantic Loss (will be activated after merge)
            if semantic_loss_fn is not None:
                eo_loss, sl_xy = semantic_loss_fn(logits)
                loss = loss + w2 * eo_loss + w3 * sl_xy

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item() * img1.size(0)
            total_samples += img1.size(0)

        train_loss = running_loss / total_samples
        val_metrics = evaluate_model(model, val_loader, semantic_loss_fn, w1, w2, w3)

        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc_x"].append(val_metrics["acc_x"])
        history["val_acc_y"].append(val_metrics["acc_y"])
        history["val_acc_o"].append(val_metrics["acc_o"])
        history["val_eo_sat"].append(val_metrics["eo_sat"])
        history["val_rule_cons"].append(val_metrics["rule_cons"])
        history["lr"].append(current_lr)

        # Save best model based on validation loss
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            torch.save(model.state_dict(), save_path)

        print(f"Epoch [{epoch:02d}/{epochs:02d}] | LR: {current_lr:.1e} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f} | "
              f"Acc (X/Y/Out): {val_metrics['acc_x']:.1f}% / {val_metrics['acc_y']:.1f}% / {val_metrics['acc_o']:.1f}% | "
              f"EO Sat: {val_metrics['eo_sat']:.1f}% | Rule Cons: {val_metrics['rule_cons']:.1f}%")

    elapsed_time = time.time() - start_time
    print(f"\nTraining completed in {elapsed_time:.2f} seconds.")
    return history, elapsed_time

# # RUN NEURAL BASELINE EXPERIMENT
# print("TRAINING NEURAL BASELINE (100% DATA)")
# baseline_model = CardNeuroSymbolicModel()
# baseline_history, baseline_time = train_model(
#     model=baseline_model,
#     train_loader=dataloaders["train_100"],
#     val_loader=dataloaders["val"],
#     epochs=10,
#     lr=1e-4,
#     semantic_loss_fn=None, # Purely supervised Cross Entropy
#     w1=1.0, w2=0.0, w3=0.0,
#     save_path="best_baseline_100.pth"
# )

# # Load best weights and evaluate on the held-out Test Set
# baseline_model.load_state_dict(torch.load("best_baseline_100.pth"))
# baseline_test_metrics = evaluate_model(baseline_model, dataloaders["test"])

# print("\nNEURAL BASELINE - FINAL TEST SET RESULTS")
# for metric_name, val in baseline_test_metrics.items():
#     print(f"{metric_name}: {val:.2f}")

Using device: cuda


###  **SEMANTIC LOSS COMPUTATION**
The network produces nine output logits, which are processed differently according to their purpose:

- **Supervised learning:** raw logits are used with standard multi-class Cross Entropy.
- **Logical regularization:** sigmoid transformations provide nine independent Bernoulli probabilities, corresponding to the Boolean propositions used by Semantic Loss.

Using independent sigmoid probabilities is essential because the Exactly-One constraints must be learned through logical regularization rather than being automatically imposed by a three-class softmax.

To compute the probability of satisfying each logical formula, we investigate **two alternative implementations of Weighted Model Counting (WMC)**:

**A. Brute-Force Enumeration**

All possible Boolean assignments are explicitly enumerated. Since there are nine propositions, the complete assignment space contains:

`2^9 = 512 possible assignments`

The satisfying assignments are identified according to the logical constraints, and their probabilities are combined to compute Semantic Loss.

**B. Sentential Decision Diagrams (SDDs)**

The logical constraints are compiled into **Sentential Decision Diagrams**, using PySDD. Weighted Model Counting is then performed on the compiled representation to calculate the probability of satisfying each logical formula.

Both implementations represent the **same logical constraints and the same mathematical Semantic Loss objective**. They differ only in the computational method used to perform Weighted Model Counting.

Both approaches are compatible with the assignment requirements. No ASP or DeepProbLog inference engine is used.

### **EVALUATION & MODEL SELECTION**

The experimental evaluation considers two complementary comparisons.

**A. Neural vs. Neuro-Symbolic**

The baseline and Semantic Loss model are evaluated using:

- **Suit Classification Accuracy:** accuracy for both card images.
- **Outcome Accuracy:** accuracy in predicting the game result.
- **Logical Constraint Satisfaction:** percentage of predictions satisfying the Exactly-One and game-rule constraints.
- **Training Time:** computational cost of model optimization.

**B. Brute Force vs. SDD**

The two Semantic Loss implementations are additionally compared in terms of:

- **Numerical Consistency:** agreement between the computed Semantic Loss values.
- **Gradient Consistency:** agreement between the gradients produced by the two implementations.
- **Computational Efficiency:** comparison of execution and computation times.

The **validation set** is used for hyperparameter selection and model configuration, while the **test set** is reserved exclusively for the final performance comparison.


**BRUTE-FORCE SEMANTIC LOSS IMPLEMENTATION**

This section implements the logical component of the neuro-symbolic model using explicit Boolean assignment enumeration and Weighted Model Counting (WMC).

The implementation is independent of the neural network: it receives nine probabilities associated with the Boolean propositions describing the two card suits and the game outcome. During development, synthetic probabilities are used to validate the calculations. Once the neural architecture is available, these inputs will be replaced by the sigmoid outputs of the network.

The implementation follows the same proposition order and logical formulas used by the PySDD implementation, so that both methods can later be compared on identical inputs.

###  1. Proposition Order and Game-Outcome Encoding

We represent the game using **nine Boolean propositions**, divided into three logical groups.
>  **Note:** The order is strictly **fixed** and must remain identical across both the *Brute-Force* and *PySDD* implementations to ensure numerical consistency.

| Index | Proposition | Meaning |
| :---: | :---: | :--- |
| **0** | `X_H` | First card is **Hearts** |
| **1** | `X_C` | First card is **Clubs** |
| **2** | `X_S` | First card is **Spades** |
| **3** | `Y_H` | Second card is **Hearts** |
| **4** | `Y_C` | Second card is **Clubs** |
| **5** | `Y_S` | Second card is **Spades** |
| **6** | `O_1` | **Player 1** wins |
| **7** | `O_x` | **Draw** |
| **8** | `O_2` | **Player 2** wins |

---
 **Outcome Encoding Convention:**
The outcome indices follow the assignment convention `{1, x, 2}`:
* `0` = Player 1 wins
* `1` = Draw
* `2` = Player 2 wins

In [12]:
import torch
from itertools import product

# Fixed proposition order shared with the PySDD implementation.
BF_PROPOSITION_ORDER = [
    "X_H", "X_C", "X_S",
    "Y_H", "Y_C", "Y_S",
    "O_1", "O_x", "O_2"
]

# All Boolean assignments for one group of three propositions: 2^3 = 8.
BF_ASSIGNMENTS_3 = torch.tensor(
    list(product((0, 1), repeat=3)),
    dtype=torch.bool
)

# All Boolean assignments for the nine propositions: 2^9 = 512.
BF_ASSIGNMENTS_9 = torch.tensor(
    list(product((0, 1), repeat=9)),
    dtype=torch.bool
)

# Rows: first-card suit (Hearts, Clubs, Spades).
# Columns: second-card suit (Hearts, Clubs, Spades).
# Values: 0 = Player 1, 1 = Draw, 2 = Player 2.
BF_EXPECTED_OUTCOME_INDEX = torch.tensor([
    [1, 0, 2],  # First card: Hearts
    [2, 1, 0],  # First card: Clubs
    [0, 2, 1]   # First card: Spades
], dtype=torch.long)

print("--- BRUTE-FORCE ASSIGNMENTS ---")
print(f"Propositions: {len(BF_PROPOSITION_ORDER)}")
print(f"Assignments for 3 propositions: {len(BF_ASSIGNMENTS_3)}")
print(f"Assignments for 9 propositions: {len(BF_ASSIGNMENTS_9)}")
print("Proposition order:", BF_PROPOSITION_ORDER)


--- BRUTE-FORCE ASSIGNMENTS ---
Propositions: 9
Assignments for 3 propositions: 8
Assignments for 9 propositions: 512
Proposition order: ['X_H', 'X_C', 'X_S', 'Y_H', 'Y_C', 'Y_S', 'O_1', 'O_x', 'O_2']


###  2. Logical Constraints and Satisfying Assignments

####  Exactly-One Constraint ($\alpha_{EO}$)
The *Exactly-One* formula is applied independently to three groups: **the first card, the second card, and the outcome**.
An assignment satisfies this constraint when **exactly one** of the three propositions in the relevant group is evaluated as true.

####  Game Rules Constraint ($\alpha_{Rules}$)
This formula is the logical conjunction ($\land$) of the nine implications specified in the assignment. For example:

$$
(X_H \land Y_C) \implies O_1
$$

>  **Logical Equivalence:**
> An implication is equivalent to `NOT premise OR conclusion` (\(A \implies B \equiv \neg A \lor B\)).
> We therefore evaluate all nine implications over the **\(2^9 = 512\) complete Boolean assignments**.

####  Satisfying Masks
The masks created in the next cell identify the satisfying assignments for each formula.
The **joint Exactly-One mask** is also retained explicitly because the *PySDD* code constructs the three Exactly-One constraints as one combined formula.

In [13]:
# ------------------------------------------------------------
# 1. EXACTLY-ONE MASK FOR A SINGLE GROUP OF THREE PROPOSITIONS
# ------------------------------------------------------------

BF_EXACTLY_ONE_MASK_3 = (BF_ASSIGNMENTS_3.sum(dim=1) == 1)


# ------------------------------------------------------------
# 2. JOINT EXACTLY-ONE MASK FOR ALL NINE PROPOSITIONS
# ------------------------------------------------------------

BF_EXACTLY_ONE_MASK_9 = (
    (BF_ASSIGNMENTS_9[:, 0:3].sum(dim=1) == 1) &
    (BF_ASSIGNMENTS_9[:, 3:6].sum(dim=1) == 1) &
    (BF_ASSIGNMENTS_9[:, 6:9].sum(dim=1) == 1)
)


# ------------------------------------------------------------
# 3. GAME-RULES MASK
# ------------------------------------------------------------

def bf_build_game_rules_mask(assignments: torch.Tensor) -> torch.Tensor:
    """Return a Boolean mask for assignments satisfying all nine rules."""
    if assignments.ndim != 2 or assignments.shape[1] != 9:
        raise ValueError("Expected assignments with shape [num_assignments, 9].")

    assignments = assignments.to(dtype=torch.bool)
    valid = torch.ones(assignments.shape[0], dtype=torch.bool, device=assignments.device)

    # for x_idx in range(3):
    #     for y_idx in range(3):
    #         # X propositions occupy columns 0..2; Y propositions 3..5.
    #         premise = assignments[:, x_idx] & assignments[:, 3 + y_idx]

    #         # Outcome propositions occupy columns 6..8.
    #         outcome_idx = int(BF_EXPECTED_OUTCOME_INDEX[x_idx, y_idx].item())
    #         expected_outcome = assignments[:, 6 + outcome_idx]

    #         # Implication: premise -> expected_outcome
    #         implication = (~premise) | expected_outcome
    #         valid &= implication

    for outcome_idx in range(3):
        premise_for_outcome = torch.zeros(
            assignments.shape[0], dtype=torch.bool, device=assignments.device
        )
        for x_idx in range(3):
            for y_idx in range(3):
                if int(BF_EXPECTED_OUTCOME_INDEX[x_idx, y_idx].item()) == outcome_idx:
                    premise_for_outcome |= (assignments[:, x_idx] & assignments[:, 3 + y_idx])

        actual_outcome = assignments[:, 6 + outcome_idx]
        valid &= (actual_outcome == premise_for_outcome)

    return valid


BF_GAME_RULES_MASK_9 = bf_build_game_rules_mask(BF_ASSIGNMENTS_9)

# Assignments satisfying both the joint Exactly-One and Game Rules formulas.
BF_JOINT_VALID_MASK_9 = BF_EXACTLY_ONE_MASK_9 & BF_GAME_RULES_MASK_9


# ------------------------------------------------------------
# 4. CHECK ASSIGNMENT COUNTS AGAINST THE LOGICAL SPECIFICATION
# ------------------------------------------------------------

bf_count_eo_single = int(BF_EXACTLY_ONE_MASK_3.sum().item())
bf_count_eo_joint = int(BF_EXACTLY_ONE_MASK_9.sum().item())
bf_count_rules = int(BF_GAME_RULES_MASK_9.sum().item())
bf_count_joint = int(BF_JOINT_VALID_MASK_9.sum().item())

print("--- BOOLEAN MODEL COUNTS ---")
print(f"Exactly-One, one group: {bf_count_eo_single} / 8")
print(f"Exactly-One, all three groups: {bf_count_eo_joint} / 512")
print(f"Game Rules only: {bf_count_rules} / 512")
print(f"Exactly-One AND Game Rules: {bf_count_joint} / 512")

assert bf_count_eo_single == 3
assert bf_count_eo_joint == 27
assert bf_count_rules == 64 #214
assert bf_count_joint == 9

print("\nAll Boolean assignment counts match the expected values.")


--- BOOLEAN MODEL COUNTS ---
Exactly-One, one group: 3 / 8
Exactly-One, all three groups: 27 / 512
Game Rules only: 64 / 512
Exactly-One AND Game Rules: 9 / 512

All Boolean assignment counts match the expected values.


###  3. Weighted Model Counting and Semantic Loss

####  Bernoulli Variables
Each of the nine propositions is treated as an independent **Bernoulli variable**.
If a proposition has probability \(p\), then its negation has probability \(1 - p\).

####  Formula Probability
The probability of a complete Boolean assignment is the **product** of the nine corresponding positive or negative literal probabilities.

The total probability of a logical formula \(\alpha\) is obtained by **summing** the probabilities of all assignments that satisfy it:

$$
P(\alpha) = \sum_{\mathbf{a} \models \alpha} \prod_{i=1}^{9} P(\text{literal}_i)
$$

####  Semantic Loss
The Semantic Loss is the negative logarithm of this probability:

$$
\mathcal{L}_{\text{Semantic}}(\alpha) = -\log \big( P(\alpha) \big)
$$

>  **Implementation Note (Numerical Stability)**
> The implementation below performs the calculation in **log space** using `torch.logsumexp`.
> This is numerically much more stable than explicitly multiplying many small probabilities and summing the resulting products.
> Furthermore, all operations remain in PyTorch, preserving the **computation graph** required for backpropagation.

In [14]:
def bf_semantic_loss_wmc_per_sample(
    probabilities: torch.Tensor,
    assignments: torch.Tensor,
    satisfying_mask: torch.Tensor,
    eps: float = 1e-7
) -> torch.Tensor:
    """Compute one Semantic Loss value per batch item using brute-force WMC.

    Args:
        probabilities: Tensor [batch_size, num_variables] with values in [0, 1].
        assignments: Boolean tensor [num_assignments, num_variables].
        satisfying_mask: Boolean tensor [num_assignments].
        eps: Numerical clamp used to avoid log(0).

    Returns:
        Tensor [batch_size] containing -log(P(formula)) for each item.
    """
    if probabilities.ndim != 2:
        raise ValueError("probabilities must have shape [batch_size, num_variables].")
    if not torch.is_floating_point(probabilities):
        raise TypeError("probabilities must be a floating-point tensor.")
    if not torch.isfinite(probabilities).all().item():
        raise ValueError("probabilities contain NaN or infinite values.")
    if ((probabilities < 0) | (probabilities > 1)).any().item():
        raise ValueError("probabilities must lie in the interval [0, 1].")
    if not (0.0 < eps < 0.5):
        raise ValueError("eps must be between 0 and 0.5.")
    if assignments.ndim != 2:
        raise ValueError("assignments must be a two-dimensional tensor.")
    if assignments.shape[1] != probabilities.shape[1]:
        raise ValueError("The number of assignment variables must match probabilities.")
    if satisfying_mask.ndim != 1 or satisfying_mask.shape[0] != assignments.shape[0]:
        raise ValueError("satisfying_mask must contain one entry per assignment.")
    if not satisfying_mask.any().item():
        raise ValueError("The logical formula has no satisfying assignments.")

    # Put the precomputed logical tensors on the same device as the inputs.
    assignments = assignments.to(device=probabilities.device, dtype=torch.bool)
    satisfying_mask = satisfying_mask.to(device=probabilities.device, dtype=torch.bool)

    # Clamp only for numerical stability; sigmoid outputs are normally inside (0, 1).
    p = probabilities.clamp(min=eps, max=1.0 - eps)
    log_p = torch.log(p).unsqueeze(1)          # [B, 1, V]
    log_not_p = torch.log1p(-p).unsqueeze(1)   # [B, 1, V]

    # For each assignment, select log(p) for True and log(1-p) for False.
    log_assignment_probabilities = torch.where(
        assignments.unsqueeze(0),
        log_p,
        log_not_p
    ).sum(dim=2)                                # [B, A]

    # Exclude assignments that do not satisfy the formula.
    log_assignment_probabilities = log_assignment_probabilities.masked_fill(
        ~satisfying_mask.unsqueeze(0),
        float("-inf")
    )

    # Log of the sum of probabilities of all satisfying assignments.
    log_formula_probability = torch.logsumexp(
        log_assignment_probabilities,
        dim=1
    )                                           # [B]

    return -log_formula_probability


def bf_semantic_loss_wmc(
    probabilities: torch.Tensor,
    assignments: torch.Tensor,
    satisfying_mask: torch.Tensor,
    eps: float = 1e-7,
    reduction: str = "mean"
) -> torch.Tensor:
    """Compute brute-force Semantic Loss with mean, sum, or no reduction."""
    per_sample = bf_semantic_loss_wmc_per_sample(
        probabilities=probabilities,
        assignments=assignments,
        satisfying_mask=satisfying_mask,
        eps=eps
    )

    if reduction == "none":
        return per_sample
    if reduction == "mean":
        return per_sample.mean()
    if reduction == "sum":
        return per_sample.sum()
    raise ValueError("reduction must be 'none', 'mean', or 'sum'.")

print("Brute-force Weighted Model Counting functions are ready.")


Brute-force Weighted Model Counting functions are ready.


###  4. Exactly-One Semantic Loss Terms

####  PySDD vs. Assignment Loss
While the `PySDD` implementation combines the three *Exactly-One* constraints into **one single formula**, the assignment loss treats them as **three separate terms**: \(EO_x\), \(EO_y\), and \(EO_o\).

####  Independent Computation
We therefore compute each term independently using the same three-variable *Exactly-One* mask.

####  Verification of the Joint Mask
We also retain the **joint Exactly-One mask** to verify a fundamental property of the loss. Specifically, we check that:

$$
\mathcal{L}_{\text{Semantic}}(EO_{\text{joint}}) = EO_x + EO_y + EO_o
$$

>  **Why does this equality hold?**
> Because the three groups (first card, second card, and outcome) contain **disjoint independent Bernoulli variables**. Mathematically, the joint probability of independent events is the product of their individual probabilities, which translates to the sum of their negative logarithms (Semantic Losses).

In [15]:
def bf_exactly_one_semantic_loss(
    group_probabilities: torch.Tensor,
    eps: float = 1e-7,
    reduction: str = "mean"
) -> torch.Tensor:
    """Exactly-One Semantic Loss for a group of three propositions."""
    if group_probabilities.ndim != 2 or group_probabilities.shape[1] != 3:
        raise ValueError("Expected group_probabilities with shape [batch_size, 3].")

    return bf_semantic_loss_wmc(
        probabilities=group_probabilities,
        assignments=BF_ASSIGNMENTS_3,
        satisfying_mask=BF_EXACTLY_ONE_MASK_3,
        eps=eps,
        reduction=reduction
    )


def bf_compute_exactly_one_terms(
    probabilities: torch.Tensor,
    eps: float = 1e-7,
    reduction: str = "mean"
) -> dict:
    """Compute EO_x, EO_y, and EO_o from a [batch_size, 9] probability tensor."""
    if probabilities.ndim != 2 or probabilities.shape[1] != 9:
        raise ValueError("Expected probabilities with shape [batch_size, 9].")

    return {
        "EO_x": bf_exactly_one_semantic_loss(probabilities[:, 0:3], eps, reduction),
        "EO_y": bf_exactly_one_semantic_loss(probabilities[:, 3:6], eps, reduction),
        "EO_o": bf_exactly_one_semantic_loss(probabilities[:, 6:9], eps, reduction),
    }


print("Exactly-One Semantic Loss functions are ready.")


Exactly-One Semantic Loss functions are ready.


###  5. Game-Rules Semantic Loss & Unified Interface

####  The Game-Rules Formula
The Game-Rules formula encapsulates the **nine logical implications** that connect every possible pair of card suits to its required game outcome.
Its satisfying Boolean assignments are identified and isolated by the pre-computed mask `BF_GAME_RULES_MASK_9`.

####  Unified Loss Interface
The next cell implements the game-rules semantic loss (\(SL_{xy}\)) and provides a **single, unified function**.
This function returns all four logical loss terms (\(EO_x\), \(EO_y\), \(EO_o\), and \(SL_{xy}\)) simultaneously, greatly simplifying the final optimization step.

>  **Training Configuration (Batch Mean)**
> The default reduction for this unified function is the **`batch mean`**. This is the standard and most suitable configuration, as it allows the logical regularizers to be seamlessly combined with the supervised Cross Entropy losses during the training loop.

In [16]:
def bf_game_rules_semantic_loss(
    probabilities: torch.Tensor,
    eps: float = 1e-7,
    reduction: str = "mean"
) -> torch.Tensor:
    """Game-Rules Semantic Loss over all nine Boolean propositions."""
    if probabilities.ndim != 2 or probabilities.shape[1] != 9:
        raise ValueError("Expected probabilities with shape [batch_size, 9].")

    return bf_semantic_loss_wmc(
        probabilities=probabilities,
        assignments=BF_ASSIGNMENTS_9,
        satisfying_mask=BF_GAME_RULES_MASK_9,
        eps=eps,
        reduction=reduction
    )


def bf_compute_semantic_loss_terms(
    probabilities: torch.Tensor,
    eps: float = 1e-7,
    reduction: str = "mean"
) -> dict:
    """Return EO_x, EO_y, EO_o, and SL_xy for a batch of nine probabilities."""
    terms = bf_compute_exactly_one_terms(
        probabilities=probabilities,
        eps=eps,
        reduction=reduction
    )
    terms["SL_xy"] = bf_game_rules_semantic_loss(
        probabilities=probabilities,
        eps=eps,
        reduction=reduction
    )
    return terms


print("Complete Brute-Force Semantic Loss interface is ready.")


Complete Brute-Force Semantic Loss interface is ready.


###  6. Numerical and Logical Validation

Before connecting this component to the neural network, we rigorously verify that the implementation agrees with both the **logical specification** and an independent **closed-form expression** for the Exactly-One probability.

####  Validation Tests
The test suite checks the following properties:

*  **Assignment Count:** The expected number of satisfying assignments for each formula.
*  **Additivity:** Equality between the joint Exactly-One loss and the sum \(EO_x + EO_y + EO_o\).
*  **Penalty Validation:** Lower loss for a high-confidence logically correct example compared to a high-confidence violation.
*  **Gradient Flow:** Finite loss values and successful gradient propagation.
*  **Autograd Verification:** Numerical gradients validated through `torch.autograd.gradcheck`.

>  **Note on Data:**
> Synthetic probabilities are used *exclusively* for these software tests. They do not represent training data or experimental model predictions.

In [17]:
# ------------------------------------------------------------
# 1. RE-CHECK THE BOOLEAN MODEL COUNTS
# ------------------------------------------------------------

assert int(BF_EXACTLY_ONE_MASK_3.sum().item()) == 3
assert int(BF_EXACTLY_ONE_MASK_9.sum().item()) == 27
assert int(BF_GAME_RULES_MASK_9.sum().item()) == 64 #214
assert int(BF_JOINT_VALID_MASK_9.sum().item()) == 9

print("Boolean model counts: 3, 27, 64, and 9 — OK")


# ------------------------------------------------------------
# 2. CREATE SYNTHETIC PROBABILITIES FOR VALIDATION
# ------------------------------------------------------------

torch.manual_seed(42)
BF_TEST_BATCH_SIZE = 4

bf_test_logits = torch.randn(
    BF_TEST_BATCH_SIZE, 9,
    dtype=torch.float64,
    requires_grad=True
)
bf_test_probabilities = torch.sigmoid(bf_test_logits)

bf_test_terms = bf_compute_semantic_loss_terms(
    bf_test_probabilities,
    reduction="none"
)

for name, value in bf_test_terms.items():
    assert torch.isfinite(value).all().item(), f"Non-finite value in {name}"
    print(f"{name}: {value.detach().mean().item():.6f} (batch mean)")


# ------------------------------------------------------------
# 3. VERIFY EXACTLY-ONE AGAINST ITS CLOSED-FORM EXPRESSION
# ------------------------------------------------------------

bf_p = torch.tensor(
    [[0.8, 0.6, 0.3]],
    dtype=torch.float64
)

bf_eo_from_enumeration = bf_exactly_one_semantic_loss(
    bf_p, reduction="none"
)

p1, p2, p3 = bf_p[0]
bf_eo_probability_closed_form = (
    p1 * (1 - p2) * (1 - p3)
    + (1 - p1) * p2 * (1 - p3)
    + (1 - p1) * (1 - p2) * p3
)
bf_eo_from_closed_form = -torch.log(bf_eo_probability_closed_form).reshape(1)

assert torch.allclose(
    bf_eo_from_enumeration,
    bf_eo_from_closed_form,
    atol=1e-10,
    rtol=1e-10
)
print("Exactly-One enumeration matches the closed-form formula — OK")


# ------------------------------------------------------------
# 4. VERIFY JOINT EXACTLY-ONE = SUM OF THREE EO TERMS
# ------------------------------------------------------------

bf_eo_terms_per_sample = bf_compute_exactly_one_terms(
    bf_test_probabilities,
    reduction="none"
)
bf_eo_sum_per_sample = (
    bf_eo_terms_per_sample["EO_x"]
    + bf_eo_terms_per_sample["EO_y"]
    + bf_eo_terms_per_sample["EO_o"]
)
bf_eo_joint_per_sample = bf_semantic_loss_wmc(
    probabilities=bf_test_probabilities,
    assignments=BF_ASSIGNMENTS_9,
    satisfying_mask=BF_EXACTLY_ONE_MASK_9,
    reduction="none"
)

assert torch.allclose(
    bf_eo_sum_per_sample,
    bf_eo_joint_per_sample,
    atol=1e-9,
    rtol=1e-9
)
print("Joint Exactly-One equals EO_x + EO_y + EO_o — OK")


# ------------------------------------------------------------
# 5. VERIFY GAME-RULES LOSS ON A CORRECT/INCORRECT EXAMPLE
# ------------------------------------------------------------

# First card Hearts, second card Clubs: Player 1 must win.
bf_correct_game = torch.tensor([[
    0.99, 0.005, 0.005,
    0.005, 0.99, 0.005,
    0.99, 0.005, 0.005
]], dtype=torch.float64)

# Same suits, but the outcome is incorrectly predicted as Draw.
bf_incorrect_game = torch.tensor([[
    0.99, 0.005, 0.005,
    0.005, 0.99, 0.005,
    0.005, 0.99, 0.005
]], dtype=torch.float64)

bf_correct_rules_loss = bf_game_rules_semantic_loss(
    bf_correct_game
)
bf_incorrect_rules_loss = bf_game_rules_semantic_loss(
    bf_incorrect_game
)

print(f"Correct game-rule loss:   {bf_correct_rules_loss.item():.6f}")
print(f"Incorrect game-rule loss: {bf_incorrect_rules_loss.item():.6f}")
assert bf_correct_rules_loss.item() < bf_incorrect_rules_loss.item()
print("Game-rule consistency test — OK")


Boolean model counts: 3, 27, 64, and 9 — OK
EO_x: 1.317169 (batch mean)
EO_y: 0.952395 (batch mean)
EO_o: 0.821428 (batch mean)
SL_xy: 2.338016 (batch mean)
Exactly-One enumeration matches the closed-form formula — OK
Joint Exactly-One equals EO_x + EO_y + EO_o — OK
Correct game-rule loss:   0.059918
Incorrect game-rule loss: 7.859646
Game-rule consistency test — OK


###  7. Backpropagation and Gradient Verification

The Semantic Loss must be **fully differentiable** so that it can effectively influence the parameters of the neural network during the training phase.

####  Verification Process
The following cell performs a two-step verification:

1. **Standard Backward Pass:** Checks that the computed gradients are both *finite* and *non-zero*, ensuring the loss is actively contributing to the optimization.
2. **Gradient Check (`gradcheck`):** Applies PyTorch's `torch.autograd.gradcheck`, which rigorously compares the automatic analytical gradients against numerical finite-difference approximations.

>  **Precision Note:**
> The test explicitly uses **double precision** (`float64`), as this is highly recommended by PyTorch for `gradcheck` to avoid floating-point rounding errors during numerical approximation.

In [18]:
from torch.autograd import gradcheck

# ------------------------------------------------------------
# 1. STANDARD BACKPROPAGATION TEST
# ------------------------------------------------------------

bf_gradient_logits = torch.randn(
    4, 9,
    dtype=torch.float64,
    requires_grad=True
)
bf_gradient_probabilities = torch.sigmoid(bf_gradient_logits)

bf_gradient_terms = bf_compute_semantic_loss_terms(
    bf_gradient_probabilities,
    reduction="mean"
)

bf_logical_test_loss = sum(bf_gradient_terms.values())
bf_logical_test_loss.backward()

assert bf_gradient_logits.grad is not None
assert torch.isfinite(bf_gradient_logits.grad).all().item()
assert bf_gradient_logits.grad.abs().sum().item() > 0

print("Backward pass: successful")
print("Gradient shape:", tuple(bf_gradient_logits.grad.shape))
print("Mean absolute gradient:", bf_gradient_logits.grad.abs().mean().item())


# ------------------------------------------------------------
# 2. NUMERICAL GRADIENT CHECK
# ------------------------------------------------------------

def bf_total_semantic_loss_for_gradcheck(logits):
    # probs = torch.sigmoid(logits)
    probs = torch.softmax(logits.reshape(-1, 3, 3), dim=-1).reshape(-1, 9)
    terms = bf_compute_semantic_loss_terms(probs, reduction="mean")
    return terms["EO_x"] + terms["EO_y"] + terms["EO_o"] + terms["SL_xy"]

bf_gradcheck_input = torch.randn(
    1, 9,
    dtype=torch.float64,
    requires_grad=True
)

bf_gradcheck_passed = gradcheck(
    bf_total_semantic_loss_for_gradcheck,
    (bf_gradcheck_input,),
    eps=1e-6,
    atol=1e-4,
    rtol=1e-3
)

print("Gradient check passed:", bf_gradcheck_passed)
assert bf_gradcheck_passed

print("\nALL BRUTE-FORCE VALIDATION TESTS PASSED")


Backward pass: successful
Gradient shape: (4, 9)
Mean absolute gradient: 0.04873108525046573
Gradient check passed: True

ALL BRUTE-FORCE VALIDATION TESTS PASSED


###  8. Interface for Future Neural-Network Integration

The Brute-Force component is now completely **decoupled** from the neural architecture. Once the CNN is available, its three groups of logits will be fed into the adapter defined below.

####  Adapter Pipeline
The adapter performs the following operations to bridge the neural and symbolic components:

1. **Sigmoid Activation:** Applies `sigmoid` to transform the raw logits into independent Bernoulli probabilities.
2. **Concatenation:** Merges the nine probabilities into a single tensor, strictly following the agreed proposition order.
3. **Loss Computation:** Computes and returns the **four Semantic Loss terms** (\(EO_x, EO_y, EO_o\), and \(SL_{xy}\)).

>  **Architectural Note:**
> No network is instantiated in this section. This adapter solely defines the **interface contract** that will be seamlessly integrated later by the neuro-symbolic training loop.

In [19]:
def bf_semantic_terms_from_logits(
    logits_x: torch.Tensor,
    logits_y: torch.Tensor,
    logits_o: torch.Tensor,
    eps: float = 1e-7,
    reduction: str = "mean"
) -> dict:
    """Adapter for future CNN outputs: three [batch_size, 3] logit tensors."""
    for name, logits in (("logits_x", logits_x), ("logits_y", logits_y), ("logits_o", logits_o)):
        if logits.ndim != 2 or logits.shape[1] != 3:
            raise ValueError(f"{name} must have shape [batch_size, 3].")

    if not (logits_x.shape[0] == logits_y.shape[0] == logits_o.shape[0]):
        raise ValueError("All three logit tensors must have the same batch size.")

    # probabilities = torch.cat(
    #     [torch.sigmoid(logits_x), torch.sigmoid(logits_y), torch.sigmoid(logits_o)],
    #     dim=1
    # )

    probabilities = torch.cat(
        [torch.softmax(logits_x, dim=1), torch.softmax(logits_y, dim=1), torch.softmax(logits_o, dim=1)],
        dim=1
    )

    return bf_compute_semantic_loss_terms(
        probabilities=probabilities,
        eps=eps,
        reduction=reduction
    )

print("Future CNN integration adapter is ready.")


Future CNN integration adapter is ready.


adattamento brute force con cnn

# ***CNN and Brute-Force Semantic Loss Integration***

The objective of this section is to connect the neural network with the brute-force Semantic Loss implementation and verify that the complete Neuro-Symbolic model works correctly.

### 1. DataLoader and CNN Compatibility

First, a batch of card image pairs is extracted from the training DataLoader and passed through the CNN.

The network must return an output tensor of shape `[batch_size, 9]`, where the nine logits represent:

- The three possible suits of the first card.
- The three possible suits of the second card.
- The three possible outcomes of the game.

Assertions are used to verify that the CNN output has the expected dimensions and is compatible with the Semantic Loss implementation.

### 2. CNN and Brute-Force Integration

The function `brute_force_semantic_loss_fn` acts as a bridge between the neural and symbolic components.

It receives the nine logits produced by the CNN, separates them into three logical groups, and passes them to the brute-force implementation, which converts them into probabilities using the sigmoid function.

The function returns two values:

- **eo_loss:** the sum of the three Exactly-One Semantic Loss terms (EO_x, EO_y, EO_o).
- **sl_xy:** the Semantic Loss associated with the game rules.

This function allows the existing training framework to use the brute-force Semantic Loss without modifying the neural architecture.

### 3. Forward Pass and Backpropagation Test

Before performing a complete training session, an integration test is conducted using a temporary CNN and a single batch of training data.

The supervised Cross Entropy Loss and the symbolic loss components are calculated and combined into the total objective function:

**L_total = w1 · L_neural + w2 · L_EO + w3 · SL_xy**

A backward pass is then performed to verify that the combined loss is finite and that finite gradients reach the neural network parameters.

This preliminary test checks the compatibility of the neural and symbolic components within the PyTorch computational graph.

### 4. Neuro-Symbolic Model Training

After verifying the integration, a new instance of the CNN is trained using the complete Neuro-Symbolic objective function.

The initial experimental configuration is:

- Training data: 100%
- Number of epochs: 20
- Learning rate: 0.0001
- Random seed: 42
- Neural Loss weight (w1): 1.0
- Exactly-One Loss weight (w2): 0.5
- Game Rules Loss weight (w3): 0.5

During training, the network learns from both labeled examples and logical constraints. The best model checkpoint is saved according to the validation loss.

The purpose of this preliminary experiment is to verify that the complete Neuro-Symbolic system can be successfully trained before conducting more systematic controlled experiments.

In [20]:
# Verifichiamo il collegamento tra DataLoader e CNN

model_test = CardNeuroSymbolicModel().to(device)

img1, img2, t_x, t_y, t_o = next(iter(dataloaders["train_100"]))

img1 = img1.to(device)
img2 = img2.to(device)

with torch.no_grad():
    test_logits = model_test(img1, img2)

print("Image 1 shape:", img1.shape)
print("Image 2 shape:", img2.shape)
print("CNN output shape:", test_logits.shape)

assert test_logits.ndim == 2
assert test_logits.shape[1] == 9

print("CNN output compatible with Semantic Loss — OK")

Image 1 shape: torch.Size([32, 3, 128, 128])
Image 2 shape: torch.Size([32, 3, 128, 128])
CNN output shape: torch.Size([32, 9])
CNN output compatible with Semantic Loss — OK


In [21]:
def brute_force_semantic_loss_fn(logits):
    """
    Connects the CNN output [batch_size, 9]
    to the brute-force Semantic Loss implementation.

    Returns:
        eo_loss: sum of the three Exactly-One losses
        sl_xy: Game Rules Semantic Loss
    """

    if logits.ndim != 2 or logits.shape[1] != 9:
        raise ValueError("CNN output must have shape [batch_size, 9].")

    # Split the nine logits into the three logical groups
    logits_x = logits[:, 0:3]
    logits_y = logits[:, 3:6]
    logits_o = logits[:, 6:9]

    # Calculate the four brute-force Semantic Loss terms
    terms = bf_semantic_terms_from_logits(
        logits_x=logits_x,
        logits_y=logits_y,
        logits_o=logits_o,
        reduction="mean"
    )

    # Combine the three Exactly-One terms
    eo_loss = (
        terms["EO_x"]
        + terms["EO_y"]
        + terms["EO_o"]
    )

    # Game Rules loss
    sl_xy = terms["SL_xy"]

    return eo_loss, sl_xy


print("CNN-Brute Force integration function ready!")

CNN-Brute Force integration function ready!


In [22]:
import torch.nn.functional as F

# Create a temporary model for the integration test
integration_model = CardNeuroSymbolicModel().to(device)
integration_model.train()

# Get one batch
img1, img2, t_x, t_y, t_o = next(iter(dataloaders["train_100"]))

img1 = img1.to(device)
img2 = img2.to(device)

t_x = t_x.to(device).long()
t_y = t_y.to(device).long()
t_o = t_o.to(device).long()

# 1. CNN forward pass
logits = integration_model(img1, img2)

# 2. Supervised classification loss
l_neural = (
    F.cross_entropy(logits[:, 0:3], t_x)
    + F.cross_entropy(logits[:, 3:6], t_y)
    + F.cross_entropy(logits[:, 6:9], t_o)
)

# 3. Brute-force Semantic Loss
eo_loss, sl_xy = brute_force_semantic_loss_fn(logits)

# 4. Combined loss (initial test weights)
w1, w2, w3 = 1.0, 0.1, 0.1

total_loss = (
    w1 * l_neural
    + w2 * eo_loss
    + w3 * sl_xy
)

print(f"Neural Loss:       {l_neural.item():.6f}")
print(f"Exactly-One Loss:  {eo_loss.item():.6f}")
print(f"Game Rules Loss:   {sl_xy.item():.6f}")
print(f"Total Loss:        {total_loss.item():.6f}")

# 5. Test backpropagation through the complete model
integration_model.zero_grad(set_to_none=True)
total_loss.backward()

gradients_exist = any(
    p.grad is not None and torch.isfinite(p.grad).all()
    for p in integration_model.parameters()
    if p.requires_grad
)

assert gradients_exist
assert torch.isfinite(total_loss).item()

print("Full CNN + Brute-force Semantic Loss integration — OK")

Neural Loss:       3.961315
Exactly-One Loss:  2.097172
Game Rules Loss:   1.678795
Total Loss:        4.338912
Full CNN + Brute-force Semantic Loss integration — OK


In [23]:
# SDD semantic loss wrapper


import time
import torch
import pandas as pd

sl_exactly_one = sl_exactly_one.to(device)
sl_game_rules = sl_game_rules.to(device)

def sdd_semantic_loss_fn(logits, eps=1e-7):
    # probs = torch.sigmoid(logits)
    probs = torch.softmax(logits.reshape(-1, 3, 3), dim=-1).reshape(-1, 9)

    _, _, eo_wmc_per_sample = sl_exactly_one(
        probabilities=probs,
        output_wmc=True,
        output_wmc_per_sample=True
    )
    _, _, rules_wmc_per_sample = sl_game_rules(
        probabilities=probs,
        output_wmc=True,
        output_wmc_per_sample=True
    )

    eo_loss = -torch.log(eo_wmc_per_sample.clamp(min=eps)).mean()
    sl_xy = -torch.log(rules_wmc_per_sample.clamp(min=eps)).mean()

    return eo_loss, sl_xy


def sdd_compute_semantic_loss_terms(probs, reduction="none", eps=1e-7):
    batch_size = probs.size(0)
    one_hot = torch.zeros((batch_size, 3), device=probs.device, dtype=probs.dtype)
    one_hot[:, 0] = 1.0  # stato [1, 0, 0] -> WMC(EO) = 1.0

    probs_x = torch.cat((probs[:, 0:3], one_hot, one_hot), dim=1)
    probs_y = torch.cat((one_hot, probs[:, 3:6], one_hot), dim=1)
    probs_o = torch.cat((one_hot, one_hot, probs[:, 6:9]), dim=1)

    _, _, eo_x_wmc = sl_exactly_one(probabilities=probs_x, output_wmc=True, output_wmc_per_sample=True)
    _, _, eo_y_wmc = sl_exactly_one(probabilities=probs_y, output_wmc=True, output_wmc_per_sample=True)
    _, _, eo_o_wmc = sl_exactly_one(probabilities=probs_o, output_wmc=True, output_wmc_per_sample=True)
    _, _, sl_xy_wmc = sl_game_rules(probabilities=probs, output_wmc=True, output_wmc_per_sample=True)

    terms = {
        "EO_x": -torch.log(eo_x_wmc.clamp(min=eps)),
        "EO_y": -torch.log(eo_y_wmc.clamp(min=eps)),
        "EO_o": -torch.log(eo_o_wmc.clamp(min=eps)),
        "SL_xy": -torch.log(sl_xy_wmc.clamp(min=eps)),
    }

    if reduction == "mean":
        return {k: v.mean() for k, v in terms.items()}
    if reduction == "sum":
        return {k: v.sum() for k, v in terms.items()}
    return terms

In [24]:
# SDD and Brute-Force comparison

import torch

with torch.no_grad():
    idx = torch.arange(512, device=device)
    all_states = ((idx.unsqueeze(1) >> torch.arange(8, -1, -1, device=device)) & 1).float()

    bf_loss = bf_compute_semantic_loss_terms(all_states, reduction="none")
    sdd_loss = sdd_compute_semantic_loss_terms(all_states, reduction="none")

    print("=== EXACT TRUTH-TABLE CHECK (512/512 states) ===")
    for name in ["EO_x", "EO_y", "EO_o", "SL_xy"]:
        bf_wmc = torch.exp(-bf_loss[name])
        sdd_wmc = torch.exp(-sdd_loss[name])

        bf_valid = torch.isclose(bf_wmc, torch.ones_like(bf_wmc), atol=1e-5)
        bf_zero = torch.isclose(bf_wmc, torch.zeros_like(bf_wmc), atol=1e-5)
        sdd_valid = torch.isclose(sdd_wmc, torch.ones_like(sdd_wmc), atol=1e-5)
        sdd_zero = torch.isclose(sdd_wmc, torch.zeros_like(sdd_wmc), atol=1e-5)

        assert torch.all(bf_valid | bf_zero), f"{name}: BF produce WMC non binari!"
        assert torch.all(sdd_valid | sdd_zero), f"{name}: SDD produce WMC non binari!"

        exact_match = torch.equal(bf_valid, sdd_valid)
        mismatches = (bf_valid != sdd_valid).sum().item()

        print(
            f"  {name:6s} | BF: {bf_valid.sum().item():3d}/512 | "
            f"SDD: {sdd_valid.sum().item():3d}/512 | "
            f"mismatches: {mismatches:3d} | EXACT MATCH: {exact_match}"
        )

=== EXACT TRUTH-TABLE CHECK (512/512 states) ===
  EO_x   | BF: 192/512 | SDD: 192/512 | mismatches:   0 | EXACT MATCH: True
  EO_y   | BF: 192/512 | SDD: 192/512 | mismatches:   0 | EXACT MATCH: True
  EO_o   | BF: 192/512 | SDD: 192/512 | mismatches:   0 | EXACT MATCH: True
  SL_xy  | BF:  64/512 | SDD:  64/512 | mismatches:   0 | EXACT MATCH: True


# **Controlled Neuro-Symbolic Experiments and Diagnostics**

The objective of this section is to evaluate the contribution of the brute-force Semantic Loss to the neural network by conducting controlled, reproducible experiments and analyzing both classification performance and logical constraint satisfaction.

### 1. Experimental Configuration and Reproducibility

All experiments share the same neural architecture, dataset, random seed, learning rate, batch size, optimizer, and number of training epochs. Each model is initialized independently under the same random conditions to ensure a fair comparison and reduce the influence of randomness on experimental results.

### 2. Data Leakage Prevention

Before training, a data leakage check is performed to verify that no image paths are shared between the training, validation, and test sets.

Since individual card images can appear in multiple generated pairs, splitting the dataset after pair generation could introduce data leakage. To prevent this issue, the original images are partitioned before generating the card pairs.

Assertions are used to stop the execution if any overlap between the three sets is detected.

### 3. Experimental Models

The experimental framework compares two main configurations:

- **Neural Baseline:** a purely neural model trained using only the supervised Cross Entropy losses.
- **Neuro-Symbolic Brute-Force:** a model trained using both supervised Cross Entropy and logical regularization through the brute-force Semantic Loss.
- **Exactly-One Only (optional):** an additional ablation experiment in which the Game Rules Loss is excluded to isolate the contribution of the Exactly-One constraints.

The general objective function is defined as:

**L_total = w1 · (L_x + L_y + L_o) + w2 · (EO_x + EO_y + EO_o) + w3 · SL_xy**

The experimental configurations differ only in the weights assigned to the logical components of the objective function.

### 4. Training and Model Selection

Each model is trained independently using the same experimental settings. During training, its performance is monitored on the validation set, and the checkpoint associated with the lowest validation loss for its respective objective function is saved.

After training, the best checkpoint is restored and evaluated using common metrics. In particular, the reported classification loss is computed using only Cross Entropy to ensure comparability between models trained with different objective functions.

### 5. Generalization Analysis

The performance of each model is evaluated on both the training and validation sets to investigate potential overfitting and underfitting.

The following evaluation metrics are considered:

- Cross Entropy Loss;
- First card classification accuracy;
- Second card classification accuracy;
- Game outcome prediction accuracy;
- Exactly-One Satisfaction;
- Game Rule Consistency;
- Total training time.

### 6. Detailed Semantic Loss Diagnostics

To better understand the contribution of symbolic knowledge, each Semantic Loss component is evaluated separately:

- **EO_x:** Exactly-One constraint for the first card;
- **EO_y:** Exactly-One constraint for the second card;
- **EO_o:** Exactly-One constraint for the game outcome;
- **SL_xy:** logical consistency between the predicted card suits and the game outcome.

Both the average probabilistic Semantic Loss and the discrete Exactly-One satisfaction rates are computed.

This distinction is important because minimizing a differentiable Semantic Loss does not necessarily guarantee that the corresponding Boolean constraints are satisfied after discretization.

### 7. Boolean Activation Analysis

The nine output logits are converted into probabilities using the sigmoid activation function and subsequently discretized using a threshold of 0.5.

For each group of propositions, the number of simultaneously active propositions is computed:

- **0 active propositions:** no class is activated;
- **1 active proposition:** the Exactly-One constraint is satisfied;
- **2 active propositions:** the Exactly-One constraint is violated;
- **3 active propositions:** the Exactly-One constraint is violated.

This analysis helps identify undesirable prediction patterns, such as the simultaneous activation of multiple classes or the absence of active propositions.

### 8. Experimental Comparison

All experimental results are collected and presented in comparative tables, allowing a systematic analysis of classification accuracy, generalization capabilities, logical constraint satisfaction, Semantic Loss components, Boolean activation distributions, and computational cost.

The test set is deliberately excluded from this experimental stage and reserved for the final evaluation after hyperparameter selection.

The ultimate goal is to investigate whether integrating symbolic knowledge into the neural learning process improves logical consistency and predictive performance, while also assessing the computational overhead introduced by brute-force Weighted Model Counting.

In [25]:
# ============================================================
# CONTROLLED NEURO-SYMBOLIC EXPERIMENTS AND DIAGNOSTICS
# ============================================================

import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import DataLoader


# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

SEED = 42
EPOCHS = 30
LR = 1e-3
BATCH_SIZE = 32

W1 = 1.0
W2 = 0.35
W3 = 0.9

# Optional additional ablation experiment
RUN_EO_ABLATION = False
USE_SDD = True
TRAIN_SPLIT = "train_100"


# ============================================================
# 2. DATA LEAKAGE CHECK
# ============================================================

manifest_dir = Path("/content/manifests")


def get_unique_images(df):
    return set(df["img1_path"]) | set(df["img2_path"])


train_images = get_unique_images(
    pd.read_csv(manifest_dir / "train_100.csv")
)

val_images = get_unique_images(
    pd.read_csv(manifest_dir / "val.csv")
)

test_images = get_unique_images(
    pd.read_csv(manifest_dir / "test.csv")
)

print("DATA LEAKAGE CHECK")
print("-" * 60)

print("Unique training images:", len(train_images))
print("Unique validation images:", len(val_images))
print("Unique test images:", len(test_images))

print("Train-Val overlap:", len(train_images & val_images))
print("Train-Test overlap:", len(train_images & test_images))
print("Val-Test overlap:", len(val_images & test_images))

assert not train_images & val_images, "Train-Val image overlap"
assert not train_images & test_images, "Train-Test image overlap"
assert not val_images & test_images, "Val-Test image overlap"

print("Image path overlap checks passed.")


# ============================================================
# 3. REPRODUCIBLE EXPERIMENT INITIALIZATION
# ============================================================

def initialize_experiment(seed=SEED, split="train_100"):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    generator = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(
        datasets[split],
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    model = CardNeuroSymbolicModel()

    return model, train_loader


# ============================================================
# 4. SEMANTIC CONSTRAINT DIAGNOSTICS
# ============================================================

def analyze_semantic_constraints(model, dataloader):

    model.eval()

    counts = {
        "EO_x_sat": 0,
        "EO_y_sat": 0,
        "EO_o_sat": 0,
        "EO_all_sat": 0
    }

    losses = {
        "EO_x": 0.0,
        "EO_y": 0.0,
        "EO_o": 0.0,
        "SL_xy": 0.0
    }

    activations = {
        "Card 1": [0, 0, 0, 0],
        "Card 2": [0, 0, 0, 0],
        "Outcome": [0, 0, 0, 0]
    }

    total = 0

    with torch.no_grad():

        for img1, img2, t_x, t_y, t_o in dataloader:

            img1 = img1.to(device)
            img2 = img2.to(device)

            logits = model(img1, img2)
            # probs = torch.sigmoid(logits)
            probs = torch.softmax(logits.reshape(-1, 3, 3), dim=-1).reshape(-1, 9)

            binary = (probs > 0.5).int()

            # Exactly-One satisfaction
            eo_x = binary[:, 0:3].sum(dim=1) == 1
            eo_y = binary[:, 3:6].sum(dim=1) == 1
            eo_o = binary[:, 6:9].sum(dim=1) == 1

            counts["EO_x_sat"] += eo_x.sum().item()
            counts["EO_y_sat"] += eo_y.sum().item()
            counts["EO_o_sat"] += eo_o.sum().item()

            counts["EO_all_sat"] += (
                eo_x & eo_y & eo_o
            ).sum().item()

            # Semantic Loss components
            # terms = bf_compute_semantic_loss_terms(
            #     probs,
            #     reduction="none"
            # )
            compute_terms_fn = sdd_compute_semantic_loss_terms if USE_SDD else bf_compute_semantic_loss_terms
            terms = compute_terms_fn(
                probs,
                reduction="none"
            )

            for name in losses:
                losses[name] += terms[name].sum().item()

            # Boolean activation distribution
            groups = {
                "Card 1": binary[:, 0:3],
                "Card 2": binary[:, 3:6],
                "Outcome": binary[:, 6:9]
            }

            for name, group in groups.items():

                active = group.sum(dim=1)

                for k in range(4):
                    activations[name][k] += (
                        active == k
                    ).sum().item()

            total += img1.size(0)

    results = {
        name: value / total * 100
        for name, value in counts.items()
    }

    for name, value in losses.items():
        results[name + "_loss"] = value / total

    activation_results = {
        name: [count / total * 100 for count in values]
        for name, values in activations.items()
    }

    return results, activation_results


# ============================================================
# 5. EXPERIMENT DEFINITIONS
# ============================================================

selected_sl_fn = sdd_semantic_loss_fn if USE_SDD else brute_force_semantic_loss_fn
print(f"Semantic Loss: {'SDD' if USE_SDD else 'Brute-Force'}")

experiments = [
    ("Baseline", 0.0, 0.0, None),
    ("Semantic-Loss", W2, W3, selected_sl_fn)
]

if RUN_EO_ABLATION:
    experiments.append(
        ("EO_Only", W2, 0.0, selected_sl_fn)
    )

models = {}
histories = {}
train_results = {}
val_results = {}
semantic_results = {}
activation_results = {}
training_times = {}


# ============================================================
# 6. MODEL TRAINING AND VALIDATION
# ============================================================

for name, w2, w3, semantic_fn in experiments:

    print("\n" + "=" * 70)
    print(f"TRAINING {name.upper()}")
    print("=" * 70)

    # model, train_loader = initialize_experiment()

    # save_path = f"best_{name}_100.pth"

    model, train_loader = initialize_experiment(split=TRAIN_SPLIT)
    save_path = f"best_{name}_{TRAIN_SPLIT}.pth"

    history, elapsed = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=dataloaders["val"],
        epochs=EPOCHS,
        lr=LR,
        semantic_loss_fn=semantic_fn,
        w1=W1,
        w2=w2,
        w3=w3,
        save_path=save_path
    )

    # Load the best checkpoint
    model.load_state_dict(
        torch.load(save_path, map_location=device)
    )

    # Evaluate using a common Cross Entropy loss
    # train_metrics = evaluate_model(
    #     model,
    #     dataloaders["train_100"]
    # )
    train_metrics = evaluate_model(
        model,
        dataloaders[TRAIN_SPLIT]
    )

    val_metrics = evaluate_model(
        model,
        dataloaders["val"]
    )

    diagnostics, activations = analyze_semantic_constraints(
        model,
        dataloaders["val"]
    )

    # Store experiment results
    models[name] = model
    histories[name] = history

    train_results[name] = train_metrics
    val_results[name] = val_metrics

    semantic_results[name] = diagnostics
    activation_results[name] = activations
    training_times[name] = elapsed


# ============================================================
# 7. GENERALIZATION ANALYSIS
# ============================================================

for name in models:

    print(f"\n{name.upper()} - GENERALIZATION ANALYSIS")
    print("-" * 65)

    for metric in train_results[name]:

        train_value = train_results[name][metric]
        val_value = val_results[name][metric]

        print(
            f"{metric:15s} | "
            f"Train: {train_value:8.4f} | "
            f"Val: {val_value:8.4f}"
        )


# ============================================================
# 8. FINAL VALIDATION COMPARISON
# ============================================================

comparison_df = pd.DataFrame(val_results)

comparison_df.loc["training_time"] = pd.Series(
    training_times
)

print("\nMODEL COMPARISON - VALIDATION SET")
display(comparison_df.round(4))


# ============================================================
# 9. DETAILED SEMANTIC LOSS COMPARISON
# ============================================================

semantic_df = pd.DataFrame(semantic_results)

print("\nSEMANTIC CONSTRAINT ANALYSIS")
display(semantic_df.round(4))


# ============================================================
# 10. BOOLEAN ACTIVATION ANALYSIS
# ============================================================

for group in ["Card 1", "Card 2", "Outcome"]:

    activation_df = pd.DataFrame({
        name: activation_results[name][group]
        for name in models
    })

    activation_df.index = [
        "0 active propositions",
        "1 active proposition",
        "2 active propositions",
        "3 active propositions"
    ]

    print(f"\nBOOLEAN ACTIVATIONS - {group}")
    display(activation_df.round(2))


# ============================================================
# END OF EXPERIMENTS
# ============================================================

print("\nAll configured experiments completed.")

DATA LEAKAGE CHECK
------------------------------------------------------------
Unique training images: 3188
Unique validation images: 546
Unique test images: 195
Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0
Image path overlap checks passed.
Semantic Loss: SDD

TRAINING BASELINE
Epoch [01/30] | LR: 1.0e-03 | Train Loss: 2.4078 | Val Loss: 1.6823 | Acc (X/Y/Out): 89.1% / 87.6% / 49.3% | EO Sat: 12.7% | Rule Cons: 50.2%
Epoch [02/30] | LR: 1.0e-03 | Train Loss: 1.5455 | Val Loss: 1.2588 | Acc (X/Y/Out): 90.0% / 88.4% / 72.7% | EO Sat: 50.2% | Rule Cons: 77.8%
Epoch [03/30] | LR: 9.9e-04 | Train Loss: 1.0935 | Val Loss: 0.9475 | Acc (X/Y/Out): 91.8% / 89.6% / 80.4% | EO Sat: 73.8% | Rule Cons: 86.9%
Epoch [04/30] | LR: 9.8e-04 | Train Loss: 0.8100 | Val Loss: 0.8924 | Acc (X/Y/Out): 91.3% / 90.4% / 83.3% | EO Sat: 89.1% | Rule Cons: 92.2%
Epoch [05/30] | LR: 9.6e-04 | Train Loss: 0.6126 | Val Loss: 0.7447 | Acc (X/Y/Out): 92.7% / 92.7% / 84.2% | EO Sat: 89.8% | Rule Cons

,Baseline,Semantic-Loss
loss,0.7090,0.6534
acc_x,93.5556,96.8889
acc_y,94.2222,96.4444
acc_o,88.0000,91.3333
eo_sat,95.1111,97.5556
rule_cons,96.0000,97.5556
training_time,77.8111,113.6354



SEMANTIC CONSTRAINT ANALYSIS


,Baseline,Semantic-Loss
EO_x_sat,99.1111,99.5556
EO_y_sat,98.4444,99.1111
EO_o_sat,96.6667,98.8889
EO_all_sat,95.1111,97.5556
EO_x_loss,0.0795,0.0414
EO_y_loss,0.0781,0.0388
EO_o_loss,0.1913,0.0662
SL_xy_loss,0.3760,0.1612



BOOLEAN ACTIVATIONS - Card 1


,Baseline,Semantic-Loss
0 active propositions,0.89,0.44
1 active proposition,99.11,99.56
2 active propositions,0.00,0.00
3 active propositions,0.00,0.00



BOOLEAN ACTIVATIONS - Card 2


,Baseline,Semantic-Loss
0 active propositions,1.56,0.89
1 active proposition,98.44,99.11
2 active propositions,0.00,0.00
3 active propositions,0.00,0.00



BOOLEAN ACTIVATIONS - Outcome


,Baseline,Semantic-Loss
0 active propositions,3.33,1.11
1 active proposition,96.67,98.89
2 active propositions,0.00,0.00
3 active propositions,0.00,0.00



All configured experiments completed.


In [26]:
# ============================================================
# 11. DATA SCARCITY SPLITS (20% & 50%) + EO_ONLY ABLATION
# ============================================================
sem_loss_fn = sdd_semantic_loss_fn if USE_SDD else brute_force_semantic_loss_fn

print("\n" + "=" * 70)
print("RUNNING DATA SCARCITY SPLITS (20%, 50%) & EO_ONLY ABLATION")
print("=" * 70)

ablation_records = []

# A. EO_Only Ablation on 100% Data (Validation Set)
print("\n--- Training EO_Only Ablation (train_100) ---")
m_eo, loader_eo = initialize_experiment(seed=SEED, split="train_100")
_, elapsed_eo = train_model(
    model=m_eo,
    train_loader=loader_eo,
    val_loader=dataloaders["val"],
    epochs=EPOCHS,
    lr=LR,
    semantic_loss_fn=sem_loss_fn,
    w1=W1, w2=W2, w3=0.0,
    save_path="best_EO_Only_100.pth"
)
m_eo.load_state_dict(torch.load("best_EO_Only_100.pth", map_location=device))
eo_val_metrics = evaluate_model(m_eo, dataloaders["val"])
eo_val_diag, _ = analyze_semantic_constraints(m_eo, dataloaders["val"])

print("\nEO_ONLY ABLATION - VALIDATION RESULTS (100% DATA):")
display(pd.DataFrame({"EO_Only_Val": {**eo_val_metrics, **eo_val_diag, "time_s": elapsed_eo}}).round(4))

# B. 20% and 50% Training Splits (Baseline vs. BruteForce) on Validation Set
for split_name in ["train_20", "train_50"]:
    for model_type, w2_cur, w3_cur, curr_fn in [
        ("Baseline", 0.0, 0.0, None),
        ("Semantic-Loss", W2, W3, sem_loss_fn)
    ]:
        print(f"\n--- Training {model_type} on {split_name} ---")
        m_sub, loader_sub = initialize_experiment(seed=SEED, split=split_name)
        ckpt_sub = f"best_{model_type}_{split_name}.pth"

        _, elapsed_sub = train_model(
            model=m_sub,
            train_loader=loader_sub,
            val_loader=dataloaders["val"],
            epochs=EPOCHS,
            lr=LR,
            semantic_loss_fn=curr_fn,
            w1=W1, w2=w2_cur, w3=w3_cur,
            save_path=ckpt_sub
        )

        m_sub.load_state_dict(torch.load(ckpt_sub, map_location=device))
        val_m = evaluate_model(m_sub, dataloaders["val"])
        val_d, _ = analyze_semantic_constraints(m_sub, dataloaders["val"])

        ablation_records.append({
            "split": split_name,
            "model": model_type,
            "val_ce_loss": val_m["loss"],
            "acc_x": val_m["acc_x"],
            "acc_y": val_m["acc_y"],
            "acc_o": val_m["acc_o"],
            "eo_sat": val_m["eo_sat"],
            "rule_cons": val_m["rule_cons"],
            "sl_xy_loss": val_d["SL_xy_loss"],
            "time_s": elapsed_sub
        })

splits_val_df = pd.DataFrame(ablation_records)
print("\nDATA SCARCITY COMPARISON (VALIDATION SET):")
display(splits_val_df.round(4))


RUNNING DATA SCARCITY SPLITS (20%, 50%) & EO_ONLY ABLATION

--- Training EO_Only Ablation (train_100) ---
Epoch [01/30] | LR: 1.0e-03 | Train Loss: 3.0026 | Val Loss: 2.1508 | Acc (X/Y/Out): 87.3% / 85.6% / 49.3% | EO Sat: 20.4% | Rule Cons: 51.8%
Epoch [02/30] | LR: 1.0e-03 | Train Loss: 1.9349 | Val Loss: 1.5286 | Acc (X/Y/Out): 91.1% / 89.8% / 76.9% | EO Sat: 54.2% | Rule Cons: 84.0%
Epoch [03/30] | LR: 9.9e-04 | Train Loss: 1.3474 | Val Loss: 1.2444 | Acc (X/Y/Out): 90.9% / 88.7% / 80.2% | EO Sat: 78.7% | Rule Cons: 88.9%
Epoch [04/30] | LR: 9.8e-04 | Train Loss: 0.9781 | Val Loss: 0.9888 | Acc (X/Y/Out): 92.7% / 92.2% / 84.4% | EO Sat: 91.3% | Rule Cons: 93.3%
Epoch [05/30] | LR: 9.6e-04 | Train Loss: 0.7543 | Val Loss: 0.9545 | Acc (X/Y/Out): 92.0% / 91.6% / 85.6% | EO Sat: 95.3% | Rule Cons: 94.9%
Epoch [06/30] | LR: 9.3e-04 | Train Loss: 0.5753 | Val Loss: 0.7959 | Acc (X/Y/Out): 92.9% / 93.6% / 87.1% | EO Sat: 94.7% | Rule Cons: 93.8%
Epoch [07/30] | LR: 9.1e-04 | Train Loss:

,EO_Only_Val
loss,0.6325
acc_x,96.8889
acc_y,95.5556
acc_o,92.2222
eo_sat,98.8889
rule_cons,98.0000
EO_x_sat,100.0000
EO_y_sat,100.0000
EO_o_sat,98.8889
EO_all_sat,98.8889



--- Training Baseline on train_20 ---
Epoch [01/30] | LR: 1.0e-03 | Train Loss: 3.2166 | Val Loss: 2.6008 | Acc (X/Y/Out): 69.3% / 69.8% / 42.2% | EO Sat: 0.0% | Rule Cons: 41.6%
Epoch [02/30] | LR: 1.0e-03 | Train Loss: 2.6119 | Val Loss: 2.3488 | Acc (X/Y/Out): 75.6% / 73.8% / 39.1% | EO Sat: 0.0% | Rule Cons: 49.6%
Epoch [03/30] | LR: 9.9e-04 | Train Loss: 2.3857 | Val Loss: 2.1497 | Acc (X/Y/Out): 83.8% / 78.2% / 49.8% | EO Sat: 7.3% | Rule Cons: 53.1%
Epoch [04/30] | LR: 9.8e-04 | Train Loss: 2.1207 | Val Loss: 1.9397 | Acc (X/Y/Out): 85.3% / 82.0% / 53.3% | EO Sat: 7.1% | Rule Cons: 58.7%
Epoch [05/30] | LR: 9.6e-04 | Train Loss: 1.8813 | Val Loss: 1.7478 | Acc (X/Y/Out): 88.7% / 84.9% / 56.2% | EO Sat: 10.0% | Rule Cons: 60.4%
Epoch [06/30] | LR: 9.3e-04 | Train Loss: 1.7025 | Val Loss: 1.6340 | Acc (X/Y/Out): 88.0% / 84.9% / 62.9% | EO Sat: 18.7% | Rule Cons: 66.4%
Epoch [07/30] | LR: 9.1e-04 | Train Loss: 1.5906 | Val Loss: 1.5297 | Acc (X/Y/Out): 90.0% / 86.7% / 64.9% | EO S

,split,model,val_ce_loss,acc_x,acc_y,acc_o,eo_sat,rule_cons,sl_xy_loss,time_s
0,train_20,Baseline,0.8828,91.5556,88.8889,83.3333,88.6667,88.8889,0.5923,20.7452
1,train_20,Semantic-Loss,0.9472,92.8889,89.7778,82.6667,95.3333,94.0000,0.4396,28.5150
2,train_50,Baseline,0.7300,94.6667,93.7778,87.3333,94.8889,96.0000,0.3238,42.7350
3,train_50,Semantic-Loss,0.6717,94.8889,92.0000,86.4444,95.5556,96.4444,0.3028,60.4631
